[To be reviewed] Completed the missing math import and sinusoidal positional-encoding forward path, including odd hidden widths. This notebook implements embedding/position components only; it does not implement or train a complete Transformer. Reference: https://arxiv.org/abs/1706.03762


In [ ]:
import math
import torch
import torch.nn as nn


In [ ]:
class InputEmbeddings(nn.Module):
    def __init__(self, d_model:int, vocab_size:int):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        return self.embedding(x)*math.sqrt(self.d_model)
    
class PositionalEncoding(nn.Module):
    """Sinusoidal encoding for batch-first inputs shaped [batch, sequence, d_model]."""

    def __init__(self, d_model: int, seq_len: int, dropout: float) -> None:
        super().__init__()
        if d_model < 1 or seq_len < 1:
            raise ValueError("d_model and seq_len must be positive")
        self.d_model = d_model
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)
        position = torch.arange(seq_len, dtype=torch.float32).unsqueeze(1)
        frequencies = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(seq_len, d_model)
        pe[:, 0::2] = torch.sin(position * frequencies)
        pe[:, 1::2] = torch.cos(position * frequencies[:d_model // 2])
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        if x.ndim != 3 or x.size(-1) != self.d_model:
            raise ValueError("Expected [batch, sequence, d_model]")
        if x.size(1) > self.seq_len:
            raise ValueError("Input exceeds the configured positional-encoding length")
        return self.dropout(x + self.pe[:, :x.size(1)].to(dtype=x.dtype))
